In [98]:
## read the dataset
dataset = open('input.txt', 'r').read()

In [110]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from collections import Counter

In [100]:
data = sorted(list(set(dataset)))

In [123]:
vocab_size = 300
initialVocab = len(data)
context_length = 6
batch_size = 4
n_embed = 10

In [102]:
## mapping (char to int) and (int, char)
stoi = {s:i for i,s in enumerate(data)}
itos = {i:s for i,s in enumerate(data)}
encoder = lambda l : [stoi[ch] for ch in l]
decoder = lambda d : [itos[id] for id in d]

In [103]:
## 20% of training data set
n3 = int(0.2 * len(dataset))
text = dataset[:n3]
## creating tokens
tokens = encoder(text)
extravocabs = vocab_size - initialVocab

In [104]:
def get_pair(tokens):

    def create_pairs(tokens):
        counter = Counter()
        for pair in zip(tokens[:], tokens[1:]):
            counter[pair]+=1
        return counter
    counter = create_pairs(tokens)
    max_pair = max(counter, key=counter.get)
    return max_pair

In [105]:
## creating vocabulary using BPE
for j in range(extravocabs):
    pair = get_pair(tokens)
    currentTokenId = j + initialVocab
    i = 0
    new_tokens = []
    while i < len(tokens):
        if i < len(tokens)-1 and (tokens[i], tokens[i+1]) == pair:
            st = itos[tokens[i]]+itos[tokens[i+1]]
            stoi[st] = currentTokenId
            itos[currentTokenId] = st
            new_tokens.append(currentTokenId)
            i+=2
        else:
            new_tokens.append(tokens[i])
            i+=1
    tokens = new_tokens

In [ ]:
## Above is the 300 size vocabulary has been created

In [113]:
## tokenize the whole dataset and split in training and validation
tokenization = torch.tensor(encoder(dataset), dtype=torch.long)
n1 = int(0.9 * len(tokenization))
train_dataset = tokenization[:n1]
val_dataset = tokenization[n1:]

In [ ]:
train_dataset

torch.Size([1003854])

In [117]:
## Next Input and output data split with the batch
torch.manual_seed(1337)
def get_batch(split):
    splitToProcess = train_dataset if split == 'train' else val_dataset
    startingPointers = torch.randint(0, len(splitToProcess)-context_length , (batch_size,))
    x = torch.stack([splitToProcess[i:i+context_length] for i in startingPointers])
    y = torch.stack([splitToProcess[i+1:i+context_length+1] for i in startingPointers])
    return x,y
    

In [118]:
x, y = get_batch('train')

In [127]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, n_embed):
        super().__init__()
        self.lookupMatrix = nn.Embedding(vocab_size, n_embed)
    def forward(self, x):
        return self.lookupMatrix(x)

In [129]:
C = TokenEmbedding(vocab_size, n_embed)
C.forward(x).shape

torch.Size([4, 6, 10])